In [ ]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import pandas as pd

os.environ['OPENAI_KEY_SAE'] = '...' # Replace with your OpenAI API key, or with another environment variable (e.g. os.environ['OPENAI_API_KEY'])

from hypothesaes.quickstart import train_sae

from hypothesaes.embedding import get_local_embeddings
from hypothesaes.sae import get_multiple_sae_activations
from hypothesaes.select_neurons import select_neurons
from hypothesaes.interpret_neurons import NeuronInterpreter, InterpretConfig, ScoringConfig, SamplingConfig, sample_percentile_bins
from hypothesaes.annotate import annotate_texts_with_concepts
from hypothesaes.evaluation import score_hypotheses

current_dir = os.getcwd()
if current_dir.endswith("notebooks"):
    prefix = "../"
else:
    prefix = "./"
base_dir = os.path.join(prefix, "data")

## Headlines

Note that Headlines requires some special handling because it is a pairwise dataset.  
Each example contains two headlines, A and B, and a binary label: 1 if headline A received higher clicks than B, 0 otherwise. A and B were randomly shuffled, so the label is approximately balanced.

We apply HypotheSAEs as follows:
1. Train SAE on all unique headlines in the train set.
2. Compute $Z_A$, an $N \times M$ activation matrix for all headline A's; similarly compute $Z_B$.
3. Compute $Z_{\Delta} = Z_A - Z_B$, which represents how much more (or less) each feature activates in headline A compared to B.
4. Select 20 neurons using L1 logistic regression on $Z_{\Delta}$.
5. Interpret selected neurons by sampling from the texts originally used to train the SAE.

For evaluation:
1. Compute $H_A$, an $N \times 20$ binary matrix of hypothesis annotations for all headlines A, and $H_B$ similarly.
2. Compute $H_{\Delta} = H_A - H_B$, and regress the label on this difference. Note that $H_{\Delta}$ can take on values in $\{-1, 0, 1\}$, which we account for in `src.evaluation.compute_hypothesis_separation_scores`.


In [100]:
DATASET_NAME = "headlines"
# EMBEDDER = "text-embedding-3-small"
# text-embedding-3-small is closed source; using this one for now
EMBEDDER = "nomic-ai/modernbert-embed-base"
CACHE_NAME = f"{DATASET_NAME}_{EMBEDDER}"

label_col = "label_pairwise"
train_df = pd.read_json(os.path.join(base_dir, DATASET_NAME, "headline_pairs_train.json"), lines=True)
val_df = pd.read_json(os.path.join(base_dir, DATASET_NAME, "headline_pairs_validation_v2.json"), lines=True)

train_labels = train_df[label_col].values

unique_train_texts = np.unique(train_df["headline_A"].tolist() + train_df["headline_B"].tolist()).tolist()
unique_val_texts = np.unique(val_df["headline_A"].tolist() + val_df["headline_B"].tolist()).tolist()

# text2embedding = get_openai_embeddings(unique_train_texts + unique_val_texts, model=EMBEDDER, cache_name=CACHE_NAME)
text2embedding = get_local_embeddings(unique_train_texts + unique_val_texts, model=EMBEDDER, cache_name=CACHE_NAME)

train_embeddings = np.stack([text2embedding[text] for text in unique_train_texts])
val_embeddings = np.stack([text2embedding[text] for text in unique_val_texts])

Loading embedding chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Loaded 16468 embeddings in 2.8s


In [101]:
checkpoint_dir = f"./checkpoints/{CACHE_NAME}"
sae_256_8 = train_sae(embeddings=train_embeddings, M=256, K=8, checkpoint_dir=checkpoint_dir, val_embeddings=val_embeddings, n_epochs=200, patience=5)
sae_32_4 = train_sae(embeddings=train_embeddings, M=32, K=4, checkpoint_dir=checkpoint_dir, val_embeddings=val_embeddings, n_epochs=200, patience=5)

sae_list = [sae_256_8, sae_32_4]

unique_train_activations, neuron_source_sae_info = get_multiple_sae_activations(sae_list, train_embeddings, return_neuron_source_info=True)

headline_A_train = np.stack([text2embedding[text] for text in train_df["headline_A"].tolist()])
headline_B_train = np.stack([text2embedding[text] for text in train_df["headline_B"].tolist()])

Z_A = get_multiple_sae_activations(sae_list, headline_A_train)
Z_B = get_multiple_sae_activations(sae_list, headline_B_train)
Z_delta = Z_A - Z_B

selected_neurons, scores = select_neurons(
    activations=Z_delta,
    target=train_labels,
    n_select=20,
    method="lasso",
    classification=True,
    verbose=True,
)

  0%|          | 0/200 [00:00<?, ?it/s]

Early stopping triggered after 79 epochs
Saved model to ./checkpoints/headlines_text-embedding-3-small/SAE_M=256_K=8.pt


  0%|          | 0/200 [00:00<?, ?it/s]

Early stopping triggered after 63 epochs
Saved model to ./checkpoints/headlines_text-embedding-3-small/SAE_M=32_K=4.pt
LASSO iteration   L1 Alpha # Features   Time (s)
----------------------------------------
       0   1.00e-01        287       1.75
       1   3.16e+01        199       0.50
       2   5.62e+02          2       0.21
       3   1.33e+02         74       0.28
       4   2.74e+02         24       0.20
       5   3.92e+02         12       0.18
       6   3.28e+02         18       0.21
       7   3.00e+02         21       0.17
       8   3.13e+02         20       0.19

Found alpha=3.13e+02 yielding exactly 20 features
Total search time: 3.70s


In [102]:
interpreter = NeuronInterpreter(
    interpreter_model="gpt-4o",
    annotator_model="gpt-4o-mini",
    n_workers_annotation=50,
    cache_name=CACHE_NAME,
)

interpretation_sampling_config = SamplingConfig(
    function=sample_percentile_bins,
    n_examples=20,
    sampling_kwargs={
        "high_percentile": (80, 100),
        "low_percentile": None, # Randomly samples from zero-activating examples
    },
)

interpret_config = InterpretConfig(
    sampling=interpretation_sampling_config,
    n_candidates=5,
    task_specific_instructions="""All of the texts are digital news headlines.
Features should describe specific characteristics in the headline texts. Example features may include:
- "explicitly mentions climate change or another environmental issue"
- "asks the reader a rhetorical question\"""",
)

interpretations = interpreter.interpret_neurons(
    texts=unique_train_texts,
    activations=unique_train_activations,
    neuron_indices=selected_neurons,
    config=interpret_config,
)

Generating 5 interpretation(s) per neuron:   0%|          | 0/100 [00:00<?, ?it/s]

In [103]:
scoring_config = ScoringConfig(
    n_examples=200,
    sampling_function=sample_percentile_bins,
    sampling_kwargs={
        "high_percentile": (80, 100),
        "low_percentile": None, # Randomly samples from zero-activating examples
    },
)

all_metrics = interpreter.score_interpretations(
    texts=unique_train_texts,
    activations=unique_train_activations,
    interpretations=interpretations,
    config=scoring_config,
)

results = []
for neuron_idx, fidelity_scores in all_metrics.items():
    best_interp = max(interpretations[neuron_idx], key=lambda interp: fidelity_scores[interp]['f1'])
    results.append({
        'neuron_idx': neuron_idx,
        'source_sae': neuron_source_sae_info[neuron_idx],
        'target_lasso_coef': scores[selected_neurons.index(neuron_idx)],
        'interpretation': best_interp,
        'f1_fidelity_score': fidelity_scores[best_interp]['f1'],
    })

hypothesis_df = pd.DataFrame(results)
pd.set_option('display.max_colwidth', None)
display(hypothesis_df.sort_values(by='target_lasso_coef', ascending=False))
pd.reset_option('display.max_colwidth')

Found 55 cached items; annotating 19835 uncached items


Scoring neuron interpretation fidelity (20 neurons; 5 candidate interps per neuron; 200 examples to score each…

,neuron_idx,source_sae,target_lasso_coef,interpretation,f1_fidelity_score
1,269,"(32, 4)",0.099923,mentions a video or visual media explicitly in the headline,0.891000
4,279,"(32, 4)",0.056853,explicitly implies the revelation of a hidden truth or secret,0.844260
5,163,"(256, 8)",0.055063,"contains words or phrases that describe something as 'awful', 'horrible', 'dangerous', 'painful', or similarly negative adjectives",0.874743
6,126,"(256, 8)",0.054179,"contains phrases that challenge the reader's assumptions or expectations (e.g., 'you might not think', 'wait till you see', 'you probably don't think', 'might change the way you look')",0.691200
7,6,"(256, 8)",0.053329,mentions a personal or emotional event followed by an unexpected or dramatic twist,0.903674
8,268,"(32, 4)",0.044655,mentions interpersonal conflict or negative interactions between individuals,0.825663
11,283,"(32, 4)",0.031996,"mentions a specific male individual using 'he', 'his', or 'him'",0.826941
14,281,"(32, 4)",0.017388,"focuses on a woman or uses female pronouns (e.g., she, her)",0.989899
15,259,"(32, 4)",0.011012,"mentions body image, appearance, or how people view themselves",0.642207
16,277,"(32, 4)",0.010464,"uses emotional language to describe personal reactions or feelings (e.g., crying, shouting, feeling sick, feeling strength)",0.883616


In [104]:
holdout_df = pd.read_json(os.path.join(base_dir, DATASET_NAME, "headline_pairs_holdout_v2.json"), lines=True)

headline_A_holdout = holdout_df["headline_A"].tolist()
headline_B_holdout = holdout_df["headline_B"].tolist()
labels_holdout = holdout_df[label_col].values

hypotheses = hypothesis_df["interpretation"].tolist()

headline_A_annotations = annotate_texts_with_concepts(
    texts=headline_A_holdout,
    concepts=hypotheses,
    cache_name=CACHE_NAME,
)

headline_B_annotations = annotate_texts_with_concepts(
    texts=headline_B_holdout,
    concepts=hypotheses,
    cache_name=CACHE_NAME,
)

delta_annots = {key : headline_A_annotations[key] - headline_B_annotations[key] for key in headline_A_annotations.keys()}

metrics, evaluation_df = score_hypotheses(
    hypothesis_annotations=delta_annots,
    y_true=labels_holdout,
    classification=True
)

pd.set_option('display.max_colwidth', None)
display(evaluation_df.sort_values(by='separation_score', ascending=False).round(4))
pd.reset_option('display.max_colwidth')

print(f"AUC: {metrics['auroc']:.3f}")
print(f"Significant hypotheses: {metrics['Significant'][0]}/{metrics['Significant'][1]} " 
      f"(p < {metrics['Significant'][2]:.3e})")

Found 0 cached items; annotating 87140 uncached items


Annotating:   0%|          | 0/87140 [00:00<?, ?it/s]

Found 5040 cached items; annotating 82100 uncached items


Annotating:   0%|          | 0/82100 [00:00<?, ?it/s]

Optimization terminated successfully.
         Current function value: 0.631850
         Iterations 5


,hypothesis,separation_score,separation_pval,regression_coef,regression_pval,feature_prevalence
7,mentions a personal or emotional event followed by an unexpected or dramatic twist,0.1812,0.0,0.4397,0.0000,0.2031
4,explicitly implies the revelation of a hidden truth or secret,0.1654,0.0,0.5535,0.0000,0.2931
18,"mentions a specific duration of time (e.g., seconds, minutes)",0.1449,0.0,0.5710,0.0001,0.0521
8,mentions interpersonal conflict or negative interactions between individuals,0.1316,0.0,0.3016,0.0002,0.1923
14,"focuses on a woman or uses female pronouns (e.g., she, her)",0.1197,0.0,0.3201,0.0010,0.1187
11,"mentions a specific male individual using 'he', 'his', or 'him'",0.1047,0.0,0.1617,0.1673,0.0842
1,mentions a video or visual media explicitly in the headline,0.1034,0.0,0.4749,0.0000,0.1047
5,"contains words or phrases that describe something as 'awful', 'horrible', 'dangerous', 'painful', or similarly negative adjectives",0.0922,0.0,0.1365,0.0867,0.1958
6,"contains phrases that challenge the reader's assumptions or expectations (e.g., 'you might not think', 'wait till you see', 'you probably don't think', 'might change the way you look')",0.0859,0.0,0.3351,0.0000,0.4044
15,"mentions body image, appearance, or how people view themselves",0.0738,0.0,0.1708,0.2466,0.0503


AUC: 0.695
Significant hypotheses: 14/20 (p < 5.000e-03)
